# MIP heuristics for the classical problems (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/mip-heuristics-classical_python.ipynb)

Executable notebook of the chapter “MIP heuristics for the classical problems” (part “Heuristic algorithms” of the course *Combinatorial Optimization*): it contains the MIP-based heuristics of the chapter (rounding, diving, the feasibility pump, RINS, local branching, proximity search, relax-and-fix and fix-and-optimize) on the models of the classical problems, and the experiment with the parameters of Gurobi, written in Python and run on every example of the chapter and on every instance of the exercises, checking each result against the expected one.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/combinatorial-optimization/heuristic-algorithms/mip-heuristics-classical/).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`: the cells must be run in order, since each one uses the functions defined above it. The first cell installs `gurobipy`, the Python interface of the solver Gurobi; its free size-limited licence is enough for every model of the notebook.

**Indices.** In the notes items, jobs, cities, machines and bins are numbered from 1, here from 0: the algorithmic logic is unchanged.

In [1]:
# install gurobipy if it is missing (in Colab): its free size-limited licence is enough for this notebook
import importlib.util, subprocess, sys
if importlib.util.find_spec("gurobipy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gurobipy"], check=True)

## Solver environment and common functions

One silent, single-thread (deterministic) Gurobi environment, and the helpers shared by all the heuristics.

In [2]:
# common.py -- solver environment and helpers shared by all the MIP-based heuristics
import math
import gurobipy as gp
from gurobipy import GRB

EPS = 1e-6                                   # integrality and comparison tolerance

ENV = gp.Env(empty=True)                     # one silent, single-thread (deterministic) environment
ENV.setParam("OutputFlag", 0)
ENV.setParam("Threads", 1)
ENV.start()


def int_index(m):
    """positions j (in m.getVars()) of the integer and binary variables: the set I"""
    return [j for j, v in enumerate(m.getVars()) if v.VType != GRB.CONTINUOUS]


def nearest(v):
    """[v] = floor(v + 1/2): rounding to the nearest integer, 1/2 rounded up"""
    return math.floor(v + 0.5 + EPS)


def is_integral(v):
    return abs(v - round(v)) <= EPS


def solve_lp(m):
    """optimal solution of the LP relaxation of m (with its current bounds), None if infeasible"""
    m.update()
    r = m.relax()
    r.optimize()
    if r.Status != GRB.OPTIMAL:
        return None
    return [v.X for v in r.getVars()]


def obj_of(m, x):
    """value of the objective function of m at the point x"""
    e = m.getObjective()
    return e.getConstant() + sum(e.getCoeff(k) * x[e.getVar(k).index] for k in range(e.size()))


def round_point(x_star, I):
    """x_tilde = [x*]: round the integer components, copy the continuous ones"""
    x_tilde = list(x_star)
    for j in I:
        x_tilde[j] = nearest(x_star[j])
    return x_tilde


def evaluate(m, x_int):
    """fix the integer variables to x_int and optimize the continuous ones:
       returns (value, x), or None if the point cannot be completed to a feasible solution"""
    f = m.copy()
    V = f.getVars()
    for j in int_index(f):
        V[j].LB = V[j].UB = x_int[j]
    f.optimize()
    if f.Status != GRB.OPTIMAL:
        return None
    return f.ObjVal, [v.X for v in V]


def solve_mip(m):
    """optimal value and solution of the MIP m, None if infeasible"""
    m.optimize()
    if m.SolCount == 0:
        return None
    return m.ObjVal, [v.X for v in m.getVars()]


def distance(m, x_tilde, I):
    """the linear expression Delta(x, x_tilde) = sum_{j in I} |x_j - x_tilde_j| in the variables of m:
       x_j - l_j if x_tilde_j is at its lower bound, u_j - x_j if at its upper bound,
       otherwise an auxiliary variable d_j >= |x_j - x_tilde_j|"""
    V = m.getVars()
    expr = gp.LinExpr()
    for j in I:
        if x_tilde[j] == V[j].LB:
            expr += V[j] - x_tilde[j]
        elif x_tilde[j] == V[j].UB:
            expr += x_tilde[j] - V[j]
        else:
            d = m.addVar(name=f"d[{j}]")
            m.addConstr(d >= V[j] - x_tilde[j])
            m.addConstr(d >= x_tilde[j] - V[j])
            expr += d
    return expr


def delta(x, x_tilde, I):
    """value of Delta(x, x_tilde)"""
    return sum(abs(x[j] - x_tilde[j]) for j in I)


def add_cutoff(m, z, theta):
    """require a solution better than z by at least theta"""
    obj = m.getObjective()
    if m.ModelSense == GRB.MINIMIZE:
        m.addConstr(obj <= z - theta, name="cutoff")
    else:
        m.addConstr(obj >= z + theta, name="cutoff")


def note(trace, *items):
    """record a step of the algorithm (used to print the examples of the notes)"""
    if trace is not None:
        trace.append(items)

## The instances

The instances of the chapter and of the exercises (0-based indices in the code).

In [3]:
# instances.py -- the instances of the notes and of the exercises (0-based indices in the code)

# reference instances
KP = dict(p=[10, 7, 6, 4], w=[5, 4, 3, 3], c=9)
BPP = dict(w=[4, 4, 5, 3, 2, 3], c=7, nb=6)
TSP = dict(d=[[0, 5, 2, 2, 9], [5, 0, 4, 3, 4], [2, 4, 0, 4, 7], [2, 3, 4, 0, 7], [9, 4, 7, 7, 0]])
PCM = dict(t=[5, 5, 4, 4, 3, 3, 3], m=3)

# heuristic solutions computed in the notes on constructive heuristics
KP_GREEDY = [1, 0, 1, 0]                       # GR: items {1,3}, profit 16
PCM_LPT = [{1, 5, 7}, {2, 6}, {3, 4}]          # LPT: sets J_1, J_2, J_3 (1-based jobs), makespan 11

# feasibility pump in two variables: max c x subject to A x <= b, 0 <= x <= U, x integer
FP2 = dict(A=[[-4, -3], [4, 5]], b=[-7, 11], c=[1, 4], U=5)

# lot sizing: the instance of the MIP course, and a capacitated one
LS_COURSE = dict(d=[20, 10, 30, 40, 10], sc=[50] * 5, vc=[0] * 5, hc=[1] * 5, cap=None)
LS_CAP = dict(d=[50, 30, 10, 10, 40, 10], sc=[40] * 6, vc=[0] * 6, hc=[1] * 6, cap=[80] * 6)


def park_miller(seed):
    """the minimal standard generator: seed = 16807 * seed mod (2^31 - 1)"""
    while True:
        seed = (16807 * seed) % 2147483647
        yield seed


def medium_bpp():
    """60 items, weights 20 + (s_i mod 41) with the Park-Miller sequence from seed 1, c = 100"""
    g = park_miller(1)
    return dict(w=[20 + next(g) % 41 for _ in range(60)], c=100, nb=24)


# exercises
EX_RD_1 = dict(p=[11, 15, 18, 14, 10], w=[3, 5, 7, 10, 9], c=16)
EX_RD_2 = dict(p=[13, 12, 14, 6, 5], w=[2, 5, 10, 6, 7], c=15)
EX_FP_1 = dict(A=[[6, -2], [-6, 7]], b=[16, -8], c=[1, 1], U=5)
EX_FP_2 = dict(A=[[-3, 5], [4, -3]], b=[11, -1], c=[4, 4], U=5)
EX_FP_3 = dict(A=[[5, 4], [-7, 3]], b=[9, 5], c=[4, 2], U=5)
EX_RL_1 = dict(p=[18, 21, 4, 19, 10, 7], w=[8, 10, 2, 12, 7, 10], c=22, x_inc=[0, 1, 0, 1, 0, 0])
EX_RL_2 = dict(p=[16, 13, 8, 16, 10, 9], w=[2, 2, 4, 9, 6, 8], c=13, x_inc=[1, 1, 1, 0, 0, 0])
EX_RL_3 = dict(t=[9, 3, 9, 5, 8, 2], m=2, sets=[{1, 4, 6}, {2, 3, 5}])

## The models

The MIP models of the classical problems, built with `gurobipy`.

In [4]:
# models.py -- the MIP models of the notes, built with gurobipy
from itertools import combinations


def build_kp(p, w, c):
    m = gp.Model("KP", env=ENV)
    n = len(p)
    x = m.addVars(n, vtype=GRB.BINARY, name="x")
    m.setObjective(gp.quicksum(p[i] * x[i] for i in range(n)), GRB.MAXIMIZE)
    m.addConstr(gp.quicksum(w[i] * x[i] for i in range(n)) <= c, name="capacity")
    m.update()
    return m


def build_bpp(w, c, nb):
    m = gp.Model("BPP", env=ENV)
    n = len(w)
    x = m.addVars(n, nb, vtype=GRB.BINARY, name="x")
    y = m.addVars(nb, vtype=GRB.BINARY, name="y")
    m.setObjective(y.sum(), GRB.MINIMIZE)
    m.addConstrs((x.sum(i, "*") == 1 for i in range(n)), name="assign")
    m.addConstrs((-gp.quicksum(w[i] * x[i, b] for i in range(n)) + c * y[b] >= 0
                  for b in range(nb)), name="capacity")
    m.update()
    return m


def build_tsp(d):
    """DFJ model: one binary variable per arc (i,j), out- and in-degree, subtour elimination constraints"""
    m = gp.Model("TSP", env=ENV)
    n = len(d)
    A = [(i, j) for i in range(n) for j in range(n) if i != j]
    x = m.addVars(A, vtype=GRB.BINARY, name="x")
    m.setObjective(gp.quicksum(d[i][j] * x[i, j] for i, j in A), GRB.MINIMIZE)
    m.addConstrs((x.sum(i, "*") == 1 for i in range(n)), name="out")
    m.addConstrs((x.sum("*", j) == 1 for j in range(n)), name="in")
    for k in range(2, n):                      # S subset of the cities except the first one, |S| >= 2
        for S in combinations(range(1, n), k):
            m.addConstr(gp.quicksum(x[i, j] for i in S for j in S if i != j) <= len(S) - 1)
    m.update()
    return m


def build_pcm(t, m):
    model = gp.Model("PCmax", env=ENV)
    n = len(t)
    x = model.addVars(n, m, vtype=GRB.BINARY, name="x")
    Cmax = model.addVar(name="Cmax")
    model.setObjective(Cmax, GRB.MINIMIZE)
    model.addConstrs((x.sum(j, "*") == 1 for j in range(n)), name="assign")
    model.addConstrs((-gp.quicksum(t[j] * x[j, h] for j in range(n)) + Cmax >= 0
                      for h in range(m)), name="load")
    model.update()
    return model


def build_lot_sizing(d, sc, vc, hc, cap=None):
    """big-M model of lot sizing: production x_t, inventory s_t, setup y_t, M_t = min{cap_t, D_tm};
       cap = None: uncapacitated; x_t and s_t continuous (with y fixed, the LP has an integer optimum)"""
    m = gp.Model("LS", env=ENV)
    T = len(d)
    y = m.addVars(T, vtype=GRB.BINARY, name="y")
    x = m.addVars(T, name="x")
    s = m.addVars(T, name="s")
    m.setObjective(gp.quicksum(sc[t] * y[t] + vc[t] * x[t] + hc[t] * s[t] for t in range(T)), GRB.MINIMIZE)
    for t in range(T):
        m.addConstr((s[t - 1] if t > 0 else 0) + x[t] == d[t] + s[t], name=f"balance[{t}]")
        M = sum(d[t:]) if cap is None else min(cap[t], sum(d[t:]))
        m.addConstr(x[t] <= M * y[t], name=f"setup[{t}]")
        if cap is not None:
            m.addConstr(x[t] <= cap[t], name=f"capacity[{t}]")
    m.addConstr(s[T - 1] == 0, name="end")     # s_0 = s_m = 0
    m.update()
    return m


def build_ip2(A, b, c, U):
    """max c1 x1 + c2 x2 subject to A x <= b, 0 <= x <= U, x integer"""
    m = gp.Model("IP2", env=ENV)
    x = m.addVars(2, lb=0, ub=U, vtype=GRB.INTEGER, name="x")
    m.setObjective(c[0] * x[0] + c[1] * x[1], GRB.MAXIMIZE)
    for k in range(len(A)):
        m.addConstr(A[k][0] * x[0] + A[k][1] * x[1] <= b[k])
    m.update()
    return m


def pcm_solution(m, sets):
    """the solution of the P||Cmax model with job sets J_1, ..., J_m (1-based jobs)"""
    x = [0] * m.NumVars
    index = {v.VarName: k for k, v in enumerate(m.getVars())}
    for h, J in enumerate(sets):
        for j in J:
            x[index[f"x[{j - 1},{h}]"]] = 1
    return evaluate(m, x)[1]

## Rounding

In [5]:
def rounding(m, trace=None):
    I = int_index(m)
    x_star = solve_lp(m)                       # optimal solution of the LP relaxation
    note(trace, "LP", x_star)
    x_tilde = round_point(x_star, I)           # x_tilde_j = [x*_j] for every integer variable
    note(trace, "rounded", x_tilde)
    return evaluate(m, x_tilde)                # (value, x) if feasible, None otherwise

## Diving

In [6]:
def diving(m, rule="fractional", x_inc=None, trace=None):
    d = m.copy()                               # the bounds of the copy are changed, not those of m
    V = d.getVars()
    I = int_index(d)
    while True:
        x_star = solve_lp(d)                   # LP relaxation with the current bounds
        if x_star is None:
            note(trace, "LP infeasible")
            return None                        # the dive fails
        F = [j for j in I if not is_integral(x_star[j])]
        note(trace, "LP", obj_of(d, x_star), x_star)
        if not F:
            return evaluate(m, round_point(x_star, I))
        if rule == "fractional":               # the least fractional variable, to its nearest integer
            j = min(F, key=lambda k: round(abs(x_star[k] - nearest(x_star[k])), 9))
            v = nearest(x_star[j])
        else:                                  # guided: the variable closest to the incumbent, to its value
            j = min(F, key=lambda k: round(abs(x_star[k] - x_inc[k]), 9))
            v = x_inc[j]
        note(trace, "fix", V[j].VarName, v)
        V[j].LB = V[j].UB = v                  # fix x_j := v

## The feasibility pump

In [7]:
def projection(m, x_tilde, I):
    """x* = argmin { Delta(x, x_tilde) : x in P }, P the LP relaxation of m"""
    p = m.relax()
    p.setObjective(distance(p, x_tilde, I), GRB.MINIMIZE)
    p.optimize()
    return [v.X for v in p.getVars()][:m.NumVars]

def feasibility_pump(m, max_iter=100, seed=1, trace=None):
    I = int_index(m)
    x_star = solve_lp(m)                       # x* = optimal solution of the LP relaxation
    note(trace, "LP", x_star)
    if all(is_integral(x_star[j]) for j in I):
        return evaluate(m, x_star)
    x_tilde = round_point(x_star, I)           # x_tilde = [x*]
    note(trace, "round", x_tilde)
    history = [x_tilde]                        # the points x_tilde visited so far
    for it in range(1, max_iter + 1):
        x_star = projection(m, x_tilde, I)     # the point of P closest to x_tilde
        dist = delta(x_star, x_tilde, I)
        note(trace, "project", x_star, dist)
        if dist <= EPS:                        # x* = x_tilde is integer and feasible
            return evaluate(m, x_tilde)
        x_new = round_point(x_star, I)
        if all(x_new[j] == x_tilde[j] for j in I):          # short cycle: [x*] = x_tilde
            j = max(I, key=lambda k: (round(abs(x_star[k] - x_tilde[k]), 9), -k))
            x_new[j] += 1 if x_star[j] > x_tilde[j] else -1  # flip the most uncertain component
            note(trace, "flip", x_new, j)
        else:
            note(trace, "round", x_new)
        if any(all(x_new[j] == y[j] for j in I) for y in history[-3:]):   # long cycle
            for j in I:                        # restart: random flips (binary variables)
                seed = (16807 * seed) % 2147483647
                rho = seed / 2147483647 - 0.3  # rho_j uniform in [-0.3, 0.7]
                if abs(x_star[j] - x_new[j]) + max(rho, 0) > 0.5:
                    x_new[j] = 1 - x_new[j]
            note(trace, "restart", x_new)
        x_tilde = x_new
        history.append(x_tilde)
    return None                                # no feasible solution found

## RINS

In [8]:
def rins(m, x_tilde, theta=1, trace=None):
    I = int_index(m)
    z_tilde = obj_of(m, x_tilde)               # value of the incumbent
    x_star = solve_lp(m)                       # optimal solution of the LP relaxation
    s = m.copy()
    V = s.getVars()
    fixed = []
    for j in I:
        if abs(x_star[j] - x_tilde[j]) <= EPS:     # LP and incumbent agree on x_j
            V[j].LB = V[j].UB = x_tilde[j]
            fixed.append(j)
    note(trace, "LP", x_star, "fixed", [V[j].VarName for j in fixed])
    add_cutoff(s, z_tilde, theta)              # look only for better solutions
    res = solve_mip(s)                         # the sub-MIP on the free variables
    if res is None:
        note(trace, "sub-MIP infeasible")
        return z_tilde, x_tilde
    note(trace, "sub-MIP", res[0])
    return res

## Local branching

In [9]:
def local_branching(m, x_tilde, k, k_max, k_step, theta=1, trace=None):
    I = int_index(m)
    w = m.copy()                               # the master MIP, which receives the reverse constraints
    z_tilde = obj_of(m, x_tilde)
    while k <= k_max:
        s = w.copy()
        s.addConstr(distance(s, x_tilde, I) <= k, name="local_branch")    # Delta(x, x_tilde) <= k
        add_cutoff(s, z_tilde, theta)
        res = solve_mip(s)
        w.addConstr(distance(w, x_tilde, I) >= k + 1)                     # never explore it again
        if res is not None:                    # better solution: re-center the neighbourhood
            note(trace, "improved", k, res[0], res[1])
            z_tilde, x_tilde = res
        else:                                  # no better solution within distance k: enlarge it
            note(trace, "no improvement", k)
            k += k_step
    return z_tilde, x_tilde

## Proximity search

In [10]:
def proximity_search(m, x_tilde, theta=1, trace=None):
    I = int_index(m)
    z_tilde = obj_of(m, x_tilde)
    while True:
        p = m.copy()
        add_cutoff(p, z_tilde, theta)          # better than the incumbent by at least theta
        p.setObjective(distance(p, x_tilde, I), GRB.MINIMIZE)   # as close as possible to it
        res = solve_mip(p)
        if res is None:                        # no solution better than z_tilde - theta exists
            note(trace, "infeasible")
            return z_tilde, x_tilde
        z_new, x_new = evaluate(m, res[1])     # value of the new solution, continuous part re-optimized
        note(trace, "found", res[0], z_new, x_new)
        z_tilde, x_tilde = z_new, x_new

## Relax-and-fix and fix-and-optimize

In [11]:
def relax_and_fix(m, blocks, trace=None):
    r = m.copy()
    V = {v.VarName: v for v in r.getVars()}
    vtype = {name: V[name].VType for B in blocks for name in B}
    for B in blocks:                           # at first, all the blocks are relaxed
        for name in B:
            V[name].VType = GRB.CONTINUOUS
    for B in blocks:
        for name in B:                         # the current block is integer again
            V[name].VType = vtype[name]
        res = solve_mip(r)
        if res is None:
            note(trace, "infeasible")
            return None
        note(trace, "block", B, res[0], {n: V[n].X for BB in blocks for n in BB})
        for name in B:                         # fix the current block to its value
            V[name].LB = V[name].UB = round(V[name].X)
    return res

In [12]:
def fix_and_optimize(m, x_tilde, windows, theta=1, trace=None):
    I = int_index(m)
    names = [v.VarName for v in m.getVars()]
    z_tilde = obj_of(m, x_tilde)
    improved = True
    while improved:                            # one pass over all the windows
        improved = False
        for W in windows:
            s = m.copy()
            V = s.getVars()
            for j in I:
                if names[j] not in W:          # outside the window: fixed to the incumbent
                    V[j].LB = V[j].UB = x_tilde[j]
            add_cutoff(s, z_tilde, theta)
            res = solve_mip(s)
            if res is not None:
                z_tilde, x_tilde = res
                improved = True
            note(trace, "window", W, z_tilde)
    return z_tilde, x_tilde

## Heuristics inside the solver

The same model solved with different values of the parameters of Gurobi that control its heuristics, with a work limit.

In [13]:
def run_with_parameters(m, params, work_limit):
    """solve m with the given Gurobi parameters for at most work_limit work units;
       returns incumbent, dual bound, nodes, work and the list of the incumbents found (work, value)"""
    s = m.copy()
    s.Params.WorkLimit = work_limit
    for name, value in params.items():
        s.setParam(name, value)
    found = []
    def callback(model, where):                # called by Gurobi at every new incumbent
        if where == GRB.Callback.MIPSOL:
            found.append((model.cbGet(GRB.Callback.WORK), model.cbGet(GRB.Callback.MIPSOL_OBJ)))
    s.optimize(callback)
    z = s.ObjVal if s.SolCount > 0 else None
    return z, s.ObjBound, int(s.NodeCount), s.Work, found

## Execution on the examples of the chapter

Every example of the chapter and every instance of the exercises, with the expected value written in the code: each line starts with `ok` when the value found is the expected one (`ERROR` otherwise); the last line counts the tests and the errors.

In [14]:
# run_tests.py -- runs every example of the notes and every exercise instance, and compares the values
# with those computed by an independent engine (exact arithmetic and enumeration, without Gurobi)

errors = 0
tests = 0


def check(name, got, expected):
    global errors, tests
    tests += 1
    if got is None or expected is None:
        good = got is expected
    else:
        good = abs(got - expected) <= 1e-6
    if not good:
        errors += 1
    print(f"{'ok   ' if good else 'ERROR'} {name:45s} got {got}  expected {expected}")


def value(res):
    return None if res is None else res[0]


def fp2(I):
    return build_ip2(I["A"], I["b"], I["c"], I["U"])


# ---------------------------------------------------------------- the four models and their LP relaxations
kp = build_kp(**KP)
bpp = build_bpp(**BPP)
tsp = build_tsp(**TSP)
pcm = build_pcm(**PCM)
for name, m, lp, opt in [("KP", kp, 17.75, 17), ("BPP", bpp, 3, 3), ("TSP", tsp, 18, 18), ("PCmax", pcm, 9, 9)]:
    check(name + " LP relaxation", obj_of(m, solve_lp(m)), lp)
    check(name + " optimum", value(solve_mip(m.copy())), opt)

# ---------------------------------------------------------------- rounding and diving
check("KP rounding", value(rounding(kp)), 16)
check("BPP rounding (infeasible)", value(rounding(bpp)), None)
check("PCmax rounding (infeasible)", value(rounding(pcm)), None)
check("TSP rounding", value(rounding(tsp)), 18)
tr = []
check("KP fractional diving", value(diving(kp, trace=tr)), 16)
check("KP diving: LP values", sum(s[1] for s in tr if s[0] == "LP"), 17.75 + 52 / 3 + 16)
tr = []
check("PCmax fractional diving", value(diving(pcm, trace=tr)), 10)
check("PCmax diving: number of fixings", len([s for s in tr if s[0] == "fix"]), 10)
check("BPP fractional diving (fails)", value(diving(bpp)), None)
x_lpt = pcm_solution(pcm, PCM_LPT)
check("LPT makespan", obj_of(pcm, x_lpt), 11)
check("PCmax guided diving from LPT", value(diving(pcm, "guided", x_lpt)), 9)
for name, I, r, d in [("EX_RD_1", EX_RD_1, 44, 44), ("EX_RD_2", EX_RD_2, None, 26)]:
    m = build_kp(**I)
    check(name + " rounding", value(rounding(m)), r)
    check(name + " fractional diving", value(diving(m)), d)

# ---------------------------------------------------------------- feasibility pump
for name, I, found, nproj in [("FP example", FP2, 5, 3), ("EX_FP_1", EX_FP_1, 4, 2),
                              ("EX_FP_2", EX_FP_2, 20, 3), ("EX_FP_3", EX_FP_3, 4, 2)]:
    tr = []
    check(name + " pump value", value(feasibility_pump(fp2(I), trace=tr)), found)
    check(name + " projections", len([s for s in tr if s[0] == "project"]), nproj)
check("KP pump", value(feasibility_pump(kp)), 16)
tr = []
check("PCmax pump", value(feasibility_pump(pcm, trace=tr)), 11)
check("PCmax pump projections", len([s for s in tr if s[0] == "project"]), 2)
for seed, bins in [(1, 4), (2, 4), (3, 4), (4, 5), (5, 5), (6, 4), (7, 3), (8, 4), (9, 4), (10, 5)]:
    check(f"BPP pump, seed {seed}", value(feasibility_pump(build_bpp(**BPP), seed=seed)), bins)

# ---------------------------------------------------------------- RINS, local branching, proximity search
check("KP RINS from GR", rins(kp, KP_GREEDY)[0], 16)
check("PCmax RINS from LPT", rins(pcm, x_lpt)[0], 9)
check("KP local branching from GR", local_branching(kp, KP_GREEDY, 1, 3, 1)[0], 17)
tr = []
check("PCmax local branching from LPT", local_branching(pcm, x_lpt, 2, 6, 2, trace=tr)[0], 9)
check("PCmax local branching: improvements", len([s for s in tr if s[0] == "improved"]), 2)
check("KP proximity search from GR", proximity_search(kp, KP_GREEDY)[0], 17)
tr = []
check("PCmax proximity search from LPT", proximity_search(pcm, x_lpt, trace=tr)[0], 9)
check("PCmax proximity: distances", sum(s[1] for s in tr if s[0] == "found"), 8)
for name, I, r, lb, ps in [("EX_RL_1", EX_RL_1, 43, 40, 43), ("EX_RL_2", EX_RL_2, 37, 45, 45)]:
    m = build_kp(I["p"], I["w"], I["c"])
    check(name + " RINS", rins(m, I["x_inc"])[0], r)
    check(name + " local branching k <= 2", local_branching(m, I["x_inc"], 1, 2, 1)[0], lb)
    check(name + " proximity search", proximity_search(m, I["x_inc"])[0], ps)
m = build_pcm(EX_RL_3["t"], EX_RL_3["m"])
x = pcm_solution(m, EX_RL_3["sets"])
check("EX_RL_3 list scheduling", obj_of(m, x), 20)
tr = []
check("EX_RL_3 proximity search", proximity_search(m, x, trace=tr)[0], 18)
check("EX_RL_3 proximity: distances", sum(s[1] for s in tr if s[0] == "found"), 6)

# ---------------------------------------------------------------- relax-and-fix, fix-and-optimize
for name, I, lp, opt, rf, fo in [("lot sizing (course)", LS_COURSE, 93.39646464646465, 170, 170, 170),
                                  ("lot sizing (capacitated)", LS_CAP, 102.38095238095238, 160, 170, 160)]:
    m = build_lot_sizing(**I)
    T = len(I["d"])
    check(name + " LP", obj_of(m, solve_lp(m)), lp)
    check(name + " optimum", value(solve_mip(m.copy())), opt)
    res = relax_and_fix(m, [[f"y[{t}]"] for t in range(T)])
    check(name + " relax-and-fix", res[0], rf)
    check(name + " fix-and-optimize", fix_and_optimize(m, res[1], [[f"y[{t}]", f"y[{t + 1}]"] for t in range(T - 1)])[0], fo)

# ---------------------------------------------------------------- heuristics inside Gurobi (medium BPP)
B = medium_bpp()
check("medium BPP total weight", sum(B["w"]), 2271)
mb = build_bpp(**B)
for name, params, z in [("default", {}, 23), ("Heuristics=0", {"Heuristics": 0}, 24),
                        ("MIPFocus=1", {"MIPFocus": 1}, 23), ("Heuristics=0.5", {"Heuristics": 0.5}, 23),
                        ("NoRelHeurWork=1", {"NoRelHeurWork": 1}, 23), ("RINS=0", {"RINS": 0}, 23)]:
    res = run_with_parameters(mb, params, 10)
    check("medium BPP, " + name, res[0], z)
    check("medium BPP, " + name + " bound", res[1], 23)

print(f"\n{tests} tests, {errors} errors")

ok    KP LP relaxation                              got 17.75  expected 17.75
ok    KP optimum                                    got 17.0  expected 17
ok    BPP LP relaxation                             got 3.0  expected 3
ok    BPP optimum                                   got 3.0  expected 3
ok    TSP LP relaxation                             got 18.0  expected 18
ok    TSP optimum                                   got 18.0  expected 18
ok    PCmax LP relaxation                           got 9.0  expected 9
ok    PCmax optimum                                 got 9.0  expected 9
ok    KP rounding                                   got 16.0  expected 16
ok    BPP rounding (infeasible)                     got None  expected None
ok    PCmax rounding (infeasible)                   got None  expected None
ok    TSP rounding                                  got 18.0  expected 18
ok    KP fractional diving                          got 16.0  expected 16
ok    KP diving: LP values            

---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE-CODE).